In [1]:
import os
import json
from datetime import datetime
import numpy as np


import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib import gridspec
import seaborn_image as isns

from scipy.constants import Boltzmann as kb
from hdr.oop_diffusion import *
from hdr.utils import *
from hdr.analytical_solution import *

import warnings

# ------------------------------------------------------------------------------------
# Retain each run in a safe place.
# ------------------------------------------------------------------------------------
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"free_diffusion_simulation/run_{timestamp}"
os.makedirs(output_dir, exist_ok=True)

# ------------------------------------------------------------------------------------
# CONSTANTS
# ------------------------------------------------------------------------------------


N = 100                                    # number of simultaneous simulations
dt = 20e-3                 # [ms]
steps = 1000                           
D = 100 * 10**(-12)     #[m^2/s]

# Can also calculate D from Stokes Einstein for Experimental Comparison
#T   = 22 + 273.15                       # K
#R_h = 50//2 * 10**(-9)                  # m
#D = kb * T / (6 * np.pi * 0.001 * R_h)  # m^2/s

print(f"Simulation Settings: \n >Particles {N} \n >Diffusion {D:.2e}")

# Save parameters
params = {
    "N": N,
    "dt": dt,
    "steps": steps,
    "D": D
}

with open(os.path.join(output_dir, "params.json"), 'w') as f:
    json.dump(params, f, indent=4)

# ------------------------------------------------------------------------------------
# Run Simulation
# ------------------------------------------------------------------------------------
sims = {}
free_sim = PeriodicCubeRandomWalkGPT(N, dt, steps, D)
free_sim.runSimulation()
free_sim.getMeanSquaredDisplacementFFT()
t_array_free = np.linspace(0, dt * steps, steps)
sims['free'] = (free_sim, t_array_free)

Simulation Settings: 
 >Particles 100 
 >Diffusion 1.00e-10


100%|██████████| 1000/1000 [00:00<00:00, 95266.63it/s]


In [11]:
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

# ------------------------------------------------------------------------------------
# PLOTTING
# ------------------------------------------------------------------------------------
fig = plt.figure(figsize=(10, 4.5), dpi=120, constrained_layout=True)

gs = gridspec.GridSpec(
    1, 2,
    width_ratios=[1.1, 1.6],
    figure=fig
)

ax1 = fig.add_subplot(gs[0, 0], projection='3d')
ax3 = fig.add_subplot(gs[0, 1])

# -------------------------------------
# Trajectory PLOT (3D, normalized)
# -------------------------------------
free_sim.plotTrajectory(fig, ax1, particle_index=0)
ax1.set_title("Trajectory", pad=10)

#ax1.set_xlim(0, 1)
#ax1.set_ylim(0, 1)
#ax1.set_zlim(0, 1)

ax1.set_xticks([])
ax1.set_yticks([])
ax1.set_zticks([])

ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.set_zlabel("")

# -------------------------------------
# FREE MSD PLOT
# -------------------------------------
free_sim, t_cyl = sims['free']
#free_sim.plotMeanMSD(fig, ax3)
time_arr = np.linspace(0, free_sim.dt * free_sim.steps, free_sim.MSD.shape[0])
analytical_line = 6 * D * time_arr

ax3.plot(time_arr, analytical_line, 'r--', label='analytical')
ax3.set_title("MSD", pad=8)
#ax3.set_xlim(0,15)
#ax3.set_ylim(0,15)
ax3.set_xlabel("Time", labelpad=6)
ax3.set_ylabel(r"$\langle x^2 \rangle$", labelpad=6)
ax3.grid(True, which='both', linestyle=':', linewidth=0.5)
ax3.legend(loc='upper right', frameon=False)

# -------------------------------------
# Displacement Histogram INSET
# -------------------------------------
XD = np.array(free_sim.TRAJ)
x = XD[:,0,0]
y = XD[:,1,0]
z = XD[:,2,0]
dx = np.diff(x)
dy = np.diff(y)
dz = np.diff(z)

ax2 = inset_axes(
    ax3,
    width="35%",
    height="35%",
    loc="upper left",
    borderpad=1.5
)

ax2.hist(dx, bins=100, alpha=0.5, label='dx')
ax2.hist(dy, bins=100, alpha=0.5, label='dy')
ax2.hist(dz, bins=100, alpha=0.5, label='dz')

ax2.set_title("Displacement PDF", fontsize=9)
ax2.tick_params(labelsize=8)
ax2.legend(frameon=False, fontsize=8)

def savefig(name):
    plt.savefig(os.path.join(OUTDIR, f"{name}.pdf"))
    plt.close()
import os
OUTDIR = "output_figures"
os.makedirs(OUTDIR, exist_ok=True)

savefig('diffusion_placeholder')
